# Leische — exploration: can context beat the target-only baseline on these labels?

A **separate, disposable** notebook. It pulls the committed pipeline in
(`leische_pipeline.ipynb` §1–§8b, executed in place — same data, same frozen
folds, same metrics), then tries a different way of *integrating* the three
context sources, on **fold 0** first and on all five folds for the best variant.
Nothing here changes the thesis pipeline or its results.

Why these changes — see [suggestions/IMPROVEMENTS.md](suggestions/IMPROVEMENTS.md)
for the literature and the full argument. In one paragraph: the committed model
encodes every context item *separately*, pools each channel to one vector and
fuses late through a gate; the labels, however, were produced by LLM annotators
that read the target **inside** its rendered thread. Every strong result on
conversational sarcasm (Ghosh et al. 2018; Dong et al. 2020; the FigLang 2020
shared task) puts the context and the target in **one** encoder input so token
attention can model the incongruity directly. Retrieval and author history are
turned into cheap leakage-safe *priors* (kNN sarcasm rate over training rows,
author sarcasm rate over training rows), and the three annotators' votes are
used as multi-annotator supervision (Davani et al. 2022) instead of being
collapsed into one label.

> Every number is agreement with the LLM-ensemble labels (Fleiss κ 0.376;
> human-vs-ensemble κ −0.148 on 31 items), never with human judgement.

Run headless: `uv run python tools/run_notebook.py --notebook leische_explore_context.ipynb`
(log to `results/logs/`). Finished variants reload from `results/explore/`.

## 1 · Pull the pipeline in (§1–§8b of leische_pipeline.ipynb, executed here)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "pyproject.toml").exists(), "run from the repo root"
sys.path.insert(0, str(ROOT / "tools"))
from run_notebook import load_cells, select   # the driver's own cell loader — no duplicated code

_pipeline = ROOT / "leische_pipeline.ipynb"
for _c in select(load_cells(_pipeline), None, ["1", "2", "3", "5", "6", "7", "8", "8b"]):
    exec(compile(_c["source"], f"<pipeline cell {_c['index']}>", "exec"), globals())
print(f"\npipeline loaded: {len(df)} rows, {len(FOLDS)} frozen folds, train-ready={GATE.passed}, "
      f"retrieval space {EMB.vectors.shape}")

## 2 · Diagnostics on fold 0 — what signal do retrieval and author history carry as *priors*?

Before building anything: if the MiniLM neighbourhood's label rate and the
author's labelled history carried no signal, feeding them in would be pointless.
Both are computed from **training-fold rows only**, same-thread excluded.

In [ ]:
from collections import Counter
from sklearn.metrics import average_precision_score, roc_auc_score

EXPLORE_FOLD = 0
FOLD = FOLDS[EXPLORE_FOLD]
Y = LABEL_MAPS["adjudicated"]
BASE_RATE = float(np.mean([Y[t] for t in FOLD["train"]]))


def knn_label_rates(fold, query_names, ks=(5, 25, 100)):
    """kNN sarcasm rate over TRAINING rows (cosine, same thread excluded — which
    also excludes the row itself), for each query. Returns {k: rate[len(query)]}."""
    train = fold["train"]
    T = EMB.vectors[[EMB.index[t] for t in train]]
    ty = np.array([Y[t] for t in train], dtype=float)
    tthr = np.array([_thread_id[_thread_of[t]] for t in train])
    out = {k: np.zeros(len(query_names)) for k in ks}
    dens = np.zeros(len(query_names))
    for s in range(0, len(query_names), 1024):
        q = query_names[s:s + 1024]
        sims = EMB.vectors[[EMB.index[x] for x in q]] @ T.T
        qthr = np.array([_thread_id[_thread_of[x]] for x in q])
        sims[qthr[:, None] == tthr[None, :]] = -np.inf
        top = np.argsort(-sims, axis=1)[:, :max(ks)]
        for k in ks:
            out[k][s:s + len(q)] = ty[top[:, :k]].mean(1)
        dens[s:s + len(q)] = np.take_along_axis(sims, top[:, :5], 1).mean(1)
    return out, dens


def author_prior(fold, query_names, smoothing=2.0):
    """Author's sarcasm rate over TRAINING-fold rows, leave-one-out for rows that
    are themselves in the training fold (a row never sees its own label)."""
    n, pos = Counter(), Counter()
    for t in fold["train"]:
        a = _rows_by_name[t]["author_hash"]; n[a] += 1; pos[a] += Y[t]
    train_set = set(fold["train"])
    cnt, rate = np.zeros(len(query_names)), np.zeros(len(query_names))
    for i, x in enumerate(query_names):
        a = _rows_by_name[x]["author_hash"]; c, p = n[a], pos[a]
        if x in train_set:
            c -= 1; p -= Y[x]
        cnt[i] = c; rate[i] = (p + smoothing * BASE_RATE) / (c + smoothing)
    return cnt, rate


test_names = FOLD["test"]
y_test = np.array([Y[t] for t in test_names])
rates, dens = knn_label_rates(FOLD, test_names)
cnt, arate = author_prior(FOLD, test_names)
print(f"fold {EXPLORE_FOLD}: {len(test_names)} test rows, base rate {y_test.mean():.3f} (= chance AUPRC)")
for k, r in rates.items():
    print(f"  kNN-{k:<3} sarcasm rate as a score: AUPRC {average_precision_score(y_test, r):.3f}  AUROC {roc_auc_score(y_test, r):.3f}")
has = cnt > 0
print(f"  author prior: {has.mean():.1%} of test rows have a labelled training row by the same author "
      f"(median {np.median(cnt[has]):.0f}); AUPRC on those {average_precision_score(y_test[has], arate[has]):.3f} "
      f"(base rate there {y_test[has].mean():.3f}), AUROC {roc_auc_score(y_test[has], arate[has]):.3f}")
print("  → both are weak but above chance: worth a few scalar features, not a channel of encoder passes.")

# what the committed pipeline scored on this fold (for the read-out at the end)
REF = {}
for name in ("ablation-1_baseline", "ablation-1_baseline-seed42", "ablation-1_baseline-seed7",
             "ablation-8_full", "ablation-8_full-seed42", "ablation-8_full-seed7", "ablation-5_conv_temp"):
    p = RESULTS / name / "fold_metrics.csv"
    if p.exists():
        fm = pd.read_csv(p); r = fm[fm["fold"] == EXPLORE_FOLD].iloc[0]
        REF[name] = dict(f1=float(r["f1"]), f1_tuned=float(r["f1_tuned"]), auprc=float(r["auprc"]), auroc=float(r["auroc"]))
if REF:
    print(f"\ncommitted pipeline on fold {EXPLORE_FOLD} (from results/):")
    print(pd.DataFrame(REF).T.round(4).to_string())

## 3 · Inputs

- **Early-fusion block** — one encoder input. `block_style="labelled"` renders
  plain role prefixes in `block_order`; `block_style="uyam"` reproduces the
  sarc-v2 annotator prompt's own rendering (`=== THREAD CONTEXT ===`,
  `[SUBMISSION r/x] title` + selftext, `[PARENT depth=d (OP)] …` for up to six
  ancestors, `[REPLY i (OP)] …` for up to three replies, then
  `=== TARGET (type) ===` **last**), from `uyam/src/uyam/annotate/context.py`. Each segment carries a role prefix and its own token
  budget; segments are joined with the encoder's separator; if the whole block
  overflows `max_len`, the longest *non-target* segment is trimmed first, so the
  target is never cut.
- **Scalar priors** (standardised on the training fold): kNN-25 / kNN-100 sarcasm
  rate, local density, author log-count and smoothed author sarcasm rate
  (leave-one-out on training rows), log-count of the author's earlier unlabelled
  posts within 48 h and unbounded.
- **Annotator votes**: the three annotator labels in a fixed model order
  (missing → −1) and their vote share.

In [ ]:
from dataclasses import dataclass, field, asdict


@dataclass
class XConfig:
    run_name: str = "x"
    encoder_name: str = "xlm-roberta-base"
    pooling: str = "mean"                  # "mean" | "cls"
    fusion: str = "early"                  # "none" = target only | "early" = one rendered block
    block_order: str = "post,parents,target,replies"   # used by block_style="labelled"
    block_style: str = "labelled"          # "labelled" (plain role prefixes) | "uyam" (the sarc-v2
                                           # prompt's own rendering: === THREAD CONTEXT ===,
                                           # [SUBMISSION r/x], [PARENT depth=d (OP)], [REPLY i],
                                           # then === TARGET (type) === LAST — what the annotators saw)
    role_prefix: bool = True
    max_len: int = 512
    budget_target: int = 192
    budget_post: int = 128
    budget_parent: int = 96
    budget_reply: int = 64
    n_parents: int = 2                     # last two ancestors
    n_replies: int = 3
    scalar_features: bool = False          # kNN prior + author prior + history counts
    annotator_heads: bool = False          # one head per annotator model (Davani et al. 2022)
    aux_weight: float = 0.5
    soft_labels: bool = False              # target = ½ shipped label + ½ annotator vote share
    lr_encoder: float = 2e-5
    lr_heads: float = 1e-4
    batch_size: int = 16
    grad_accum: int = 2
    max_epochs: int = 10
    patience: int = 3
    dropout: float = 0.2
    mlp_hidden: int = 256
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0
    seed: int = 13
    fold: int = 0
    label_source: str = "adjudicated"

    def to_dict(self):
        return asdict(self)


ROLE_PREFIX = {"post": "POST: ", "parent": "PARENT: ", "target": "COMMENT: ", "reply": "REPLY: "}
ANNOTATOR_ORDER = ("gemma3", "qwen3", "sealion")


def block_parts(row, xc: XConfig):
    ctx = row["context"]
    parts = {"post": [], "parents": [], "target": [("target", row["text"], xc.budget_target)], "replies": []}
    sub = ctx.get("submission")
    if sub:
        text = " ".join(t for t in (sub.get("title"), sub.get("selftext")) if t)
        if text.strip():
            parts["post"] = [("post", text, xc.budget_post)]
    parents = [p for p in (ctx.get("parent_chain") or []) if (p.get("text") or "").strip()][-xc.n_parents:]
    parts["parents"] = [("parent", p["text"], xc.budget_parent) for p in parents]
    replies = [p for p in (ctx.get("replies") or []) if (p.get("text") or "").strip()][:xc.n_replies]
    parts["replies"] = [("reply", p["text"], xc.budget_reply) for p in replies]
    return [p for key in (k.strip() for k in xc.block_order.split(",")) for p in parts[key]]


def block_parts_uyam(row, xc: XConfig):
    """The sarc-v2 annotator rendering (uyam/src/uyam/annotate/context.py + prompts.py):
    context block first, target last, same markers. Texts in the snapshot are already
    the char-truncated ones the annotators saw; the token budgets here only cap them."""
    ctx, sub, parts = row["context"], row["context"].get("submission"), []
    sr = row.get("subreddit") or ""
    if row["record_type"] == "comment":
        if sub:
            head = f"=== THREAD CONTEXT ===\n[SUBMISSION r/{sr}] {sub.get('title') or ''}"
            if (sub.get("selftext") or "").strip():
                head += "\n" + sub["selftext"].strip()
        else:
            head = f"=== THREAD CONTEXT ===\n[SUBMISSION r/{sr}] (submission not collected)"
    else:
        head = f"=== THREAD CONTEXT ===\n[SUBMISSION r/{sr}] (the TARGET below is this submission)"
    parts.append(("post", head, xc.budget_post))
    for p_ in (ctx.get("parent_chain") or [])[-xc.n_parents:]:
        if (p_.get("text") or "").strip():
            marker = " (OP)" if p_.get("is_submitter") else ""
            parts.append(("parent", f"[PARENT depth={p_.get('depth')}{marker}] {p_['text']}", xc.budget_parent))
    for i, r_ in enumerate((ctx.get("replies") or [])[:xc.n_replies], start=1):
        if (r_.get("text") or "").strip():
            marker = " (OP)" if r_.get("is_submitter") else ""
            parts.append(("reply", f"[REPLY {i}{marker}] {r_['text']}", xc.budget_reply))
    parts.append(("target", f"=== TARGET ({row['record_type']}) ===\n{row['text']}", xc.budget_target))
    return parts


class BlockEncoder:
    """Token-level assembly so every segment keeps its own budget and the target is never truncated."""

    def __init__(self, xc: XConfig):
        self.xc = xc
        self.tok = AutoTokenizer.from_pretrained(xc.encoder_name)
        self.bos, self.sep, self.pad = self.tok.cls_token_id, self.tok.sep_token_id, self.tok.pad_token_id
        assert None not in (self.bos, self.sep, self.pad), "tokenizer lacks cls/sep/pad ids"

    def _ids(self, text, budget):
        return self.tok(text, add_special_tokens=False, truncation=True, max_length=budget)["input_ids"]

    def encode(self, row):
        xc = self.xc
        if xc.fusion == "none":
            return [self.bos] + self._ids(row["text"], xc.budget_target) + [self.sep]
        if xc.block_style == "uyam":
            pieces = [(role, self._ids(text, budget)) for role, text, budget in block_parts_uyam(row, xc)]
        else:
            pieces = [(role, self._ids((ROLE_PREFIX[role] if xc.role_prefix else "") + text, budget))
                      for role, text, budget in block_parts(row, xc)]
        total = 1 + sum(len(p) for _, p in pieces) + len(pieces)
        while total > xc.max_len:
            j = max((i for i, (r, _) in enumerate(pieces) if r != "target"), key=lambda i: len(pieces[i][1]), default=None)
            if j is None or len(pieces[j][1]) <= 8:
                break
            role, ids = pieces[j]; pieces[j] = (role, ids[:-8]); total -= 8
        out = [self.bos]
        for i, (_, ids) in enumerate(pieces):
            if i:
                out.append(self.sep)
            out.extend(ids)
        out.append(self.sep)
        return out[:xc.max_len]


_ENC_CACHE = {}


def encode_all(xc: XConfig, names):
    key = (xc.encoder_name, xc.fusion, xc.block_style, xc.block_order, xc.role_prefix, xc.max_len, xc.budget_target,
           xc.budget_post, xc.budget_parent, xc.budget_reply, xc.n_parents, xc.n_replies)
    if key not in _ENC_CACHE:
        enc = BlockEncoder(xc)
        t0 = time.time()
        _ENC_CACHE[key] = ({f: enc.encode(_rows_by_name[f]) for f in df["reddit_fullname"]}, enc.pad)
        lens = np.array([len(v) for v in _ENC_CACHE[key][0].values()])
        print(f"encoded {len(lens)} rows for {xc.encoder_name} ({xc.fusion}): median {np.median(lens):.0f} tokens, "
              f"p90 {np.percentile(lens, 90):.0f}, max {lens.max()} [{time.time() - t0:.0f}s]")
    ids, pad = _ENC_CACHE[key]
    return {f: ids[f] for f in names}, pad


_FEAT_CACHE = {}


def scalar_features(fold_i):
    """Leakage-safe priors for every row of a fold, standardised on its training rows."""
    if fold_i in _FEAT_CACHE:
        return _FEAT_CACHE[fold_i]
    fold = FOLDS[fold_i]
    names = fold["train"] + fold["val"] + fold["test"]
    rates, dens = knn_label_rates(fold, names, ks=(25, 100))
    cnt, arate = author_prior(fold, names)
    t48 = build_temporal(Config(temporal_k=5, temporal_window_hours=48.0))
    tinf = build_temporal(Config(temporal_k=10, temporal_window_hours=None))
    X = np.stack([rates[25], rates[100], dens, np.log1p(cnt), arate,
                  np.log1p([len(t48[x]) for x in names]), np.log1p([len(tinf[x]) for x in names])], 1)
    _train_set = set(fold["train"])
    tr = np.array([x in _train_set for x in names])
    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6
    X = (X - mu) / sd
    _FEAT_CACHE[fold_i] = {x: X[i].astype(np.float32) for i, x in enumerate(names)}
    return _FEAT_CACHE[fold_i]


FEATURE_NAMES = ["knn25_rate", "knn100_rate", "knn_density", "author_logn", "author_rate", "hist48_logn", "hist_logn"]


def annotator_votes(row):
    by = {a["model_key"]: a.get("sarcastic") for a in (row["reliability"].get("annotators") or [])}
    votes = [int(by[m]) if isinstance(by.get(m), bool) else -1 for m in ANNOTATOR_ORDER]
    have = [v for v in votes if v >= 0]
    return votes, (sum(have) / len(have) if have else float(row["labels"]["sarcastic"]))


print("input builders ready")

## 4 · Model and training loop (bf16 autocast, early stopping on validation F1, best weights restored)

In [ ]:
class FusedModel(nn.Module):
    def __init__(self, xc: XConfig, n_feats: int):
        super().__init__()
        self.xc = xc
        backbone = AutoModel.from_pretrained(xc.encoder_name)
        self.encoder = nn.Module()             # build_optimizer expects model.encoder.backbone
        self.encoder.backbone = backbone
        h = backbone.config.hidden_size
        self.feat_proj = (nn.Sequential(nn.Linear(n_feats, 32), nn.GELU())
                          if xc.scalar_features and n_feats else None)
        d = h + (32 if self.feat_proj is not None else 0)
        self.classifier = nn.Sequential(nn.Dropout(xc.dropout), nn.Linear(d, xc.mlp_hidden), nn.GELU(),
                                        nn.Dropout(xc.dropout), nn.Linear(xc.mlp_hidden, 2))
        self.annot_heads = nn.ModuleList([nn.Linear(d, 2) for _ in ANNOTATOR_ORDER]) if xc.annotator_heads else None

    def forward(self, batch):
        hs = self.encoder.backbone(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state
        if self.xc.pooling == "mean":
            m = batch["attention_mask"].unsqueeze(-1).to(hs.dtype)
            h = (hs * m).sum(1) / m.sum(1).clamp(min=1e-6)
        else:
            h = hs[:, 0]
        if self.feat_proj is not None:
            h = torch.cat([h, self.feat_proj(batch["feats"].to(h.dtype))], dim=-1)
        out = {"logits": self.classifier(h), "features": h, "target_emb": h, "gates": None}
        if self.annot_heads is not None:
            out["annot_logits"] = torch.stack([head(h) for head in self.annot_heads], dim=1)   # [B, 3, 2]
        return out


class BlockDataset(Dataset):
    def __init__(self, names, ids, feats, labels):
        self.names, self.ids, self.feats, self.labels = list(names), ids, feats, labels

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        f = self.names[i]
        votes, share = annotator_votes(_rows_by_name[f])
        return {"fullname": f, "ids": self.ids[f], "feats": self.feats[f] if self.feats else np.zeros(1, np.float32),
                "label": int(self.labels[f]), "annot": votes, "share": share}


def collate_blocks(samples, pad_id):
    width = max(len(s["ids"]) for s in samples)
    ids = torch.full((len(samples), width), pad_id, dtype=torch.long)
    att = torch.zeros(len(samples), width, dtype=torch.long)
    for i, s in enumerate(samples):
        ids[i, :len(s["ids"])] = torch.tensor(s["ids"]); att[i, :len(s["ids"])] = 1
    y = torch.tensor([s["label"] for s in samples], dtype=torch.long)
    return {"fullnames": [s["fullname"] for s in samples], "input_ids": ids, "attention_mask": att,
            "feats": torch.tensor(np.stack([s["feats"] for s in samples])), "labels": y,
            "annot": torch.tensor([s["annot"] for s in samples], dtype=torch.long),
            "soft": 0.5 * y.float() + 0.5 * torch.tensor([s["share"] for s in samples])}


class BucketSampler(torch.utils.data.Sampler):
    """Length-bucketed batches (shuffled within and across buckets each epoch): early-fusion
    blocks vary 30–512 tokens, so padding to the batch maximum would waste most of the compute."""

    def __init__(self, lengths, batch_size, seed):
        self.lengths, self.bs, self.rng = np.asarray(lengths), batch_size, np.random.default_rng(seed)

    def __iter__(self):
        idx = np.argsort(self.lengths + self.rng.random(len(self.lengths)) * 8)   # small jitter
        batches = [idx[i:i + self.bs].tolist() for i in range(0, len(idx), self.bs)]
        self.rng.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return math.ceil(len(self.lengths) / self.bs)


def make_block_loaders(xc: XConfig):
    fold = FOLDS[xc.fold]
    names = fold["train"] + fold["val"] + fold["test"]
    ids, pad = encode_all(xc, names)
    feats = scalar_features(xc.fold) if xc.scalar_features else None
    labels = LABEL_MAPS[xc.label_source]
    collate = lambda s: collate_blocks(s, pad)
    loaders = {}
    for part in ("train", "val", "test"):
        ds = BlockDataset(fold[part], ids, feats, labels)
        lengths = [len(ids[f]) for f in fold[part]]
        if part == "train":
            loaders[part] = DataLoader(ds, batch_sampler=BucketSampler(lengths, xc.batch_size, xc.seed), collate_fn=collate)
        else:
            order = np.argsort(lengths)
            loaders[part] = DataLoader(ds, batch_sampler=[order[i:i + xc.batch_size * 2].tolist()
                                                          for i in range(0, len(order), xc.batch_size * 2)], collate_fn=collate)
    return loaders


def fused_loss(out, batch, xc: XConfig, class_w):
    logits, y = out["logits"], batch["labels"]
    if xc.soft_labels:
        t = batch["soft"]
        per = -(torch.stack([1 - t, t], -1) * F.log_softmax(logits, -1)).sum(-1) * class_w[y]
    else:
        per = F.cross_entropy(logits, y, weight=class_w, reduction="none")
    loss = per.mean()
    if xc.annotator_heads:
        al = batch["annot"]; mask = al >= 0
        if mask.any():
            loss = loss + xc.aux_weight * F.cross_entropy(out["annot_logits"][mask], al[mask], weight=class_w)
    return loss


def predict_blocks(model, loader):
    model.eval(); rows = []
    with torch.no_grad():
        for batch in loader:
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                out = model(batch)
            logits = out["logits"].float()
            margin = (logits[:, 1] - logits[:, 0]).cpu().numpy()
            prob = 1 / (1 + np.exp(-margin))
            ann = (F.softmax(out["annot_logits"].float(), -1)[:, :, 1].mean(1).cpu().numpy()
                   if "annot_logits" in out else None)
            for i, f in enumerate(batch["fullnames"]):
                r = {"reddit_fullname": f, "y_true": int(batch["labels"][i]), "prob": float(prob[i]),
                     "margin": float(margin[i]), "pred": int(prob[i] >= 0.5)}
                if ann is not None:
                    r["prob_annotators"] = float(ann[i])
                rows.append(r)
    return pd.DataFrame(rows)


def train_blocks(model, loaders, xc: XConfig, class_w, log=True):
    model.to(DEVICE)
    opt = build_optimizer(model, Config(lr_encoder=xc.lr_encoder, lr_heads=xc.lr_heads))
    steps = len(loaders["train"])
    sched = build_scheduler(opt, max(1, steps * xc.max_epochs // xc.grad_accum), xc.warmup_ratio)
    w = torch.tensor(class_w, dtype=torch.float, device=DEVICE)
    best, best_state, left = -1.0, None, xc.patience
    hist = {"train_loss": [], "val_f1": [], "epoch_sec": []}
    for epoch in range(xc.max_epochs):
        t0 = time.time(); model.train(); losses = []
        opt.zero_grad(set_to_none=True)
        for step, batch in enumerate(loaders["train"]):
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = fused_loss(model(batch), batch, xc, w) / xc.grad_accum
            loss.backward()
            if (step + 1) % xc.grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), xc.grad_clip)
                opt.step(); opt.zero_grad(set_to_none=True); sched.step()
            losses.append(float(loss.item()) * xc.grad_accum)
        val = predict_blocks(model, loaders["val"])
        f1 = safe_f1(val["y_true"].to_numpy(), val["pred"].to_numpy())
        hist["train_loss"].append(float(np.mean(losses))); hist["val_f1"].append(f1); hist["epoch_sec"].append(round(time.time() - t0, 1))
        if log:
            print(f"    epoch {epoch}: loss {hist['train_loss'][-1]:.4f} val_f1@0.5 {f1:.4f} ({hist['epoch_sec'][-1]:.0f}s)", flush=True)
        if f1 > best:
            best, left, best_state = f1, xc.patience, copy.deepcopy(model.state_dict())
        else:
            left -= 1
            if left <= 0:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    hist.update(best_val_f1=best, best_epoch=int(np.argmax(hist["val_f1"])) + 1, epochs_run=len(hist["val_f1"]))
    return hist


EXPLORE_DIR = RESULTS / "explore"
EXPLORE_DIR.mkdir(parents=True, exist_ok=True)
_X_KEYS = [k for k in XConfig.__dataclass_fields__ if k != "run_name"]


def run_variant(xc: XConfig, verbose=True):
    """Train one variant on its fold; skip-if-done keyed on the config. Returns one summary row."""
    out_dir = EXPLORE_DIR / f"{xc.run_name}-f{xc.fold}-s{xc.seed}"
    rj = out_dir / "run.json"
    if rj.exists():
        saved = json.loads(rj.read_text(encoding="utf-8"))
        # a knob added later is compared at its default, so older runs stay valid
        _defaults = XConfig().to_dict()
        if {k: saved["config"].get(k, _defaults[k]) for k in _X_KEYS} == {k: xc.to_dict()[k] for k in _X_KEYS}:
            print(f"{xc.run_name} f{xc.fold} s{xc.seed}: loaded from {out_dir.name}")
            return saved["summary"]
    t0 = time.time()
    set_seed(xc.seed)
    torch.cuda.reset_peak_memory_stats()
    loaders = make_block_loaders(xc)
    model = FusedModel(xc, len(FEATURE_NAMES))
    w = class_weights(FOLDS[xc.fold]["train"], LABEL_MAPS[xc.label_source])
    hist = train_blocks(model, loaders, xc, w, log=verbose)
    val = predict_blocks(model, loaders["val"])
    thr, _ = best_threshold(val["y_true"].to_numpy(), val["prob"].to_numpy())
    temp = fit_temperature(val["margin"].to_numpy(), val["y_true"].to_numpy())
    pred = predict_blocks(model, loaders["test"])
    pred["pred_tuned"] = (pred["prob"] >= thr).astype(int)
    pred["prob_cal"] = 1 / (1 + np.exp(-pred["margin"] / temp))
    pred = pred.merge(META, on="reddit_fullname", how="left")
    y = pred["y_true"].to_numpy()
    m, mt = safe_metrics(y, pred["pred"].to_numpy()), safe_metrics(y, pred["pred_tuned"].to_numpy())
    summary = {"run": xc.run_name, "fold": xc.fold, "seed": xc.seed, "encoder": xc.encoder_name, "fusion": xc.fusion,
               "feats": xc.scalar_features, "heads": xc.annotator_heads, "soft": xc.soft_labels,
               "f1": m["f1"], "precision": m["precision"], "recall": m["recall"], "f1_tuned": mt["f1"],
               "threshold": thr, **rank_metrics(y, pred["prob"].to_numpy()), "ece": ece(y, pred["prob"].to_numpy()),
               "best_val_f1": hist["best_val_f1"], "best_epoch": hist["best_epoch"], "epochs_run": hist["epochs_run"],
               "min": round((time.time() - t0) / 60, 1), "peak_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 1)}
    if "prob_annotators" in pred:
        summary["f1_annotator_avg"] = safe_metrics(y, (pred["prob_annotators"] >= 0.5).astype(int).to_numpy())["f1"]
    out_dir.mkdir(parents=True, exist_ok=True)
    pred.to_csv(out_dir / "predictions.csv", index=False)
    (out_dir / "run.json").write_text(json.dumps({"config": xc.to_dict(), "summary": summary, "history": hist,
                                                  "dataset_identity": IDENTITY, "label_authority": LABEL_AUTHORITY}, indent=2),
                                      encoding="utf-8")
    if verbose:
        print(f"{xc.run_name} f{xc.fold} s{xc.seed}: F1@0.5 {m['f1']:.4f} (P {m['precision']:.3f} R {m['recall']:.3f}) "
              f"F1@val-thr {mt['f1']:.4f} AUPRC {summary['auprc']:.4f} AUROC {summary['auroc']:.4f} "
              f"best epoch {hist['best_epoch']}/{hist['epochs_run']} | {summary['min']} min, peak {summary['peak_gb']} GB", flush=True)
    del model, loaders
    torch.cuda.empty_cache()
    return summary


print("model + loop ready")

## 5 · Variants on fold 0, seed 13

Each row changes one thing against `x0-target-only`, which re-runs the
committed baseline *inside this loop* (bf16, bucketed batches) so the
comparison is like-for-like. Budget ≈ 5–10 min per base-encoder run.

In [ ]:
VARIANTS = [
    XConfig(run_name="x0-target-only", fusion="none"),
    XConfig(run_name="x1-early-fusion"),
    XConfig(run_name="x2-early+priors", scalar_features=True),
    XConfig(run_name="x3-early+annotator-heads", annotator_heads=True),
    XConfig(run_name="x4-early+soft-votes", soft_labels=True),
    XConfig(run_name="x5-early+priors+heads+soft", scalar_features=True, annotator_heads=True, soft_labels=True),
    XConfig(run_name="x6-early-target-first", block_order="target,post,parents,replies"),
    XConfig(run_name="x7-early-xlmr-large", encoder_name="xlm-roberta-large", batch_size=8, grad_accum=4, lr_encoder=1e-5),
    XConfig(run_name="x8-early-mdeberta-base", encoder_name="microsoft/mdeberta-v3-base"),
    XConfig(run_name="x9-early-roberta-tagalog", encoder_name="jcblaise/roberta-tagalog-base"),
    # attribution: x5 minus one ingredient each
    XConfig(run_name="x10-early+heads+soft", annotator_heads=True, soft_labels=True),
    XConfig(run_name="x11-early+priors+soft", scalar_features=True, soft_labels=True),
    XConfig(run_name="x12-early+priors+heads", scalar_features=True, annotator_heads=True),
    # the full recipe on the large encoder
    XConfig(run_name="x13-early-xlmr-large+priors+heads+soft", encoder_name="xlm-roberta-large", batch_size=8, grad_accum=4,
            lr_encoder=1e-5, scalar_features=True, annotator_heads=True, soft_labels=True),
    # P8: the sarc-v2 prompt's own rendering (target last, uyam markers, ≤6 parents, ≤3 replies)
    XConfig(run_name="x14-uyam-block+heads+soft", block_style="uyam", n_parents=6, n_replies=3,
            annotator_heads=True, soft_labels=True),
    XConfig(run_name="x15-uyam-block", block_style="uyam", n_parents=6, n_replies=3),
]
rows = []
for xc in VARIANTS:
    try:
        rows.append(run_variant(xc))
    except torch.OutOfMemoryError:
        print(f"{xc.run_name}: OOM under the VRAM cap — skipped"); torch.cuda.empty_cache()
    except Exception as exc:   # a missing encoder must not stop the grid
        print(f"{xc.run_name}: FAILED — {type(exc).__name__}: {str(exc)[:200]}"); torch.cuda.empty_cache()
grid = pd.DataFrame(rows)
cols = ["run", "encoder", "f1", "precision", "recall", "f1_tuned", "auprc", "auroc", "ece", "best_val_f1", "best_epoch", "min", "peak_gb"]
print("\nfold 0, seed 13 — agreement with the LLM-ensemble labels:")
print(grid[[c for c in cols if c in grid]].round(4).to_string(index=False))
grid.to_csv(EXPLORE_DIR / "grid-fold0-seed13.csv", index=False)

## 6 · Seeds for the base-encoder variants that looked best on **validation**

Picked by validation F1 (never by the test fold), plus the target-only
reference at the same seeds, so the comparison is 3 paired runs each.

In [ ]:
base = grid[(grid["encoder"] == "xlm-roberta-base") & (grid["run"] != "x0-target-only")]
TOP = base.sort_values("best_val_f1", ascending=False)["run"].head(2).tolist()
print("seeding:", TOP, "(chosen on validation F1) + x0-target-only")
seed_rows = list(rows)
for xc in VARIANTS:
    if xc.run_name in TOP or xc.run_name == "x0-target-only":
        for s in (42, 7):
            xc_s = XConfig(**{**xc.to_dict(), "seed": s})
            seed_rows.append(run_variant(xc_s))
seeded = pd.DataFrame(seed_rows)
pooled = seeded.groupby("run").agg(runs=("f1", "size"), f1=("f1", "mean"), f1_std=("f1", "std"),
                                   f1_tuned=("f1_tuned", "mean"), auprc=("auprc", "mean"), auprc_std=("auprc", "std"),
                                   auroc=("auroc", "mean"), ece=("ece", "mean")).round(4)
print("\nfold 0 — pooled over available seeds:")
print(pooled.to_string())
# paired per-seed deltas vs the target-only reference
ref = seeded[seeded["run"] == "x0-target-only"].set_index("seed")
for run in TOP:
    cur = seeded[seeded["run"] == run].set_index("seed")
    common = sorted(set(cur.index) & set(ref.index))
    d = cur.loc[common, "f1"] - ref.loc[common, "f1"]
    da = cur.loc[common, "auprc"] - ref.loc[common, "auprc"]
    print(f"  {run:<30} vs target-only over seeds {common}: ΔF1 {d.mean():+.4f} ± {d.std():.4f} (wins {(d > 0).sum()}/{len(d)}) "
          f"| ΔAUPRC {da.mean():+.4f} ± {da.std():.4f}")
seeded.to_csv(EXPLORE_DIR / "grid-fold0-seeds.csv", index=False)

## 7 · Five-fold confirmation of the best base-encoder variant (seed 13)

The one number that is directly comparable with the committed pipeline's
5-fold matrix (`results/ablation-matrix.csv`: baseline 0.353 ± 0.034 at seed
13, 0.368 ± 0.026 over three seeds).

In [ ]:
BEST = pooled.loc[[r for r in pooled.index if r in TOP]].sort_values("f1_tuned", ascending=False).index[0]
best_xc = next(v for v in VARIANTS if v.run_name == BEST)
print(f"5-fold confirmation for {BEST}")
cv_rows = []
for fi in range(len(FOLDS)):
    for run_xc in (XConfig(**{**best_xc.to_dict(), "fold": fi}), XConfig(**{**VARIANTS[0].to_dict(), "fold": fi})):
        cv_rows.append(run_variant(run_xc))
cv = pd.DataFrame(cv_rows)
summ = cv.groupby("run").agg(folds=("f1", "size"), f1=("f1", "mean"), f1_std=("f1", "std"), f1_tuned=("f1_tuned", "mean"),
                             precision=("precision", "mean"), recall=("recall", "mean"),
                             auprc=("auprc", "mean"), auprc_std=("auprc", "std"), auroc=("auroc", "mean"), ece=("ece", "mean")).round(4)
print("\n5 folds × seed 13 — this notebook's loop (bf16, bucketed):")
print(summ.to_string())
a = cv[cv["run"] == BEST].set_index("fold")["f1"]; b = cv[cv["run"] == "x0-target-only"].set_index("fold")["f1"]
print(f"\npaired per-fold ΔF1 ({BEST} − target-only): {(a - b).mean():+.4f} ± {(a - b).std():.4f}, wins {(a > b).sum()}/5; "
      f"per fold: " + " ".join(f"{v:+.3f}" for v in (a - b)))
cv.to_csv(EXPLORE_DIR / "cv-best-vs-target-only.csv", index=False)

# significance on pooled test predictions (same machinery as §12 of the pipeline)
pa = pd.concat([pd.read_csv(EXPLORE_DIR / f"x0-target-only-f{fi}-s13" / "predictions.csv").assign(seed=13) for fi in range(5)])
pb = pd.concat([pd.read_csv(EXPLORE_DIR / f"{BEST}-f{fi}-s13" / "predictions.csv").assign(seed=13) for fi in range(5)])
m = pa[["reddit_fullname", "seed", "y_true", "pred"]].merge(pb[["reddit_fullname", "seed", "pred"]], on=["reddit_fullname", "seed"], suffixes=("_a", "_b"))
rng = np.random.default_rng(13); y, xa, xb = m["y_true"].to_numpy(), m["pred_a"].to_numpy(), m["pred_b"].to_numpy()
obs = f1_score(y, xb, zero_division=0) - f1_score(y, xa, zero_division=0)
diffs = np.array([f1_score(y[i], xb[i], zero_division=0) - f1_score(y[i], xa[i], zero_division=0)
                  for i in (rng.integers(0, len(y), len(y)) for _ in range(1000))])
print(f"paired bootstrap over {len(y):,} test predictions: ΔF1 {obs:+.4f}, CI95 [{np.percentile(diffs, 2.5):+.4f}, {np.percentile(diffs, 97.5):+.4f}], "
      f"p={min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())):.3f}")

## 8 · Read-out

Interpretation, literature and the recommended changes to the thesis pipeline
are written up in [suggestions/IMPROVEMENTS.md](suggestions/IMPROVEMENTS.md).
The tables above are the evidence; every number is agreement with the
LLM-ensemble labels, never with human judgement.